In [ ]:
#bu versiyonda 30 run için bir seed listesi oluşturuldu. tüm deneyler bu seed listesinde çalıştı ve metriklere SSE eklendi

In [ ]:
!pip install -q k-means-constrained==0.9.1 #ver enviroment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 9.0 MB/s eta 0:00:00


In [ ]:
#!pip install numpy==1.26.4 scipy==1.13.0 scikit-learn==1.4.2
#!pip install k-means-constrained==0.7.3 makaledeki ver1 enviroment

In [ ]:
import pandas as pd
import random
import numpy as np
import sys
import k_means_constrained
from k_means_constrained import KMeansConstrained
from sklearn.metrics import davies_bouldin_score,calinski_harabasz_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.metrics import adjusted_mutual_info_score
from google.colab import drive
from scipy.spatial.distance import cdist
import time as time
from sklearn.metrics import adjusted_rand_score
import numpy as np
import random
from time import perf_counter


# Rastgelelik için sabit bir seed değeri belirleyin
#SEED = 42

# NumPy için rastgelelik sabitleme
#np.random.seed(SEED)

# Python'un random modülü için sabitleme
#random.seed(SEED)
#import torch  # Eğer PyTorch kullanıyorsanız



# Eğer TensorFlow veya PyTorch kullanıyorsanız:
#torch.manual_seed(SEED)



In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
from scipy.spatial.distance import cdist
def calculate_invalid_constraint(df):
    # Her kümede en az bir 'constraint' değeri 1 olan olup olmadığını kontrol et
    cluster_groups = df.groupby("assigned_cluster")["constraint"].sum()

    # En az bir 'constraint' = 1 içermeyen kümeler
    invalid_clusters = cluster_groups[cluster_groups == 0].index

    # Oran hesaplama
    total_clusters = df["assigned_cluster"].nunique()
    invalid_cluster_ratio = len(invalid_clusters) / total_clusters

    return invalid_cluster_ratio
def calculate_capacity_violation(df, cluster_capacity):

    # Her kümeye atanan nokta sayısını hesapla
    cluster_counts = df["assigned_cluster"].value_counts()

    # Kapasite ihlali yapan kümeleri bul
    violated_clusters = cluster_counts[cluster_counts > cluster_capacity].index

    # Oran hesaplama
    total_clusters = df["assigned_cluster"].nunique()
    violation_ratio = len(violated_clusters) / total_clusters

    return violation_ratio
def dunn_index(X, labels):
    unique_labels = np.unique(labels)
    min_intercluster_dist = np.inf  # En küçük kümeler arası mesafe
    max_intracluster_dist = 0       # En büyük küme içi mesafe

    # Kümeler arasındaki en küçük mesafeyi bul
    for i in unique_labels:
        for j in unique_labels:
            if i != j:
                cluster_i = X[labels == i]
                cluster_j = X[labels == j]
                inter_dist = np.min(cdist(cluster_i, cluster_j))  # Küme merkezleri değil, noktalar arasındaki min mesafe
                min_intercluster_dist = min(min_intercluster_dist, inter_dist)

    # Kümeler içindeki en büyük mesafeyi bul
    for i in unique_labels:
        cluster_i = X[labels == i]
        intra_dist = np.max(cdist(cluster_i, cluster_i))  # Küme içindeki en büyük mesafe
        max_intracluster_dist = max(max_intracluster_dist, intra_dist)

    return min_intercluster_dist / max_intracluster_dist
def calculate_sse(X, labels):
    X = np.asarray(X)
    labels = np.asarray(labels)

    sse = 0.0

    for cluster_id in np.unique(labels):
        cluster_points = X[labels == cluster_id]

        if len(cluster_points) == 0:
            continue

        centroid = np.mean(cluster_points, axis=0)

        sse += np.sum(
            (cluster_points - centroid) ** 2
        )

    return float(sse)

In [ ]:
dataset_list = ['appendicitis',
                'breast_cancer',
                'bupa',
                'ecoli',
                'glass',
                'haberman',
                'hayes-roth',
                'heart',
                'ionosphere',
                'iris',
                'led7digit',
                'monk-2',
                'movement_libras',
                'newthyroid',
                'saheart',
                'sonar',
                'soybean',
                'spectfheart',
                'tae',
                'vehicle',
                'zoo']
percentage_list=[5,10,15,20]
n_run=30

In [ ]:
#KMEANS

df_test=pd.read_excel('/content/drive/MyDrive/Real_Test/Real_ConsCluster_Test_Character.xlsx',skiprows=1)
df_test = df_test.iloc[:, 1:]
kmeans_constraint_result=[]
result_column_list=['TestNo','Run_No','nCluster','Percentage_Constraint','RunTime','Davies-Bouldin','Calinski-Harabasz','Dunn','Silhoutte','Adjusted Mutual Information (AMI)','Adjusted Rand Index (ARI)','invalid_constraint_rate','invalid_capacity_rate','sse','seed']
#kmeans_constraint_result.append(result_column_list)

SEEDS = list(range(30))
for test_no in dataset_list:
  for percentage in percentage_list:
    for run_no in range(1,n_run+1):
      seed = SEEDS[run_no - 1]

      df = pd.read_csv('/content/drive/MyDrive/Test/Real_Test_v2/modified_data_ver2/'+str(test_no)+'_data_'+str(percentage)+'.csv')
      true_labels=df['class']
      df.drop(columns=['class'],inplace=True)
      df2 = df.drop(['constraint'], axis=1)
      nCluster= df_test.loc[df_test["Test Name"] == test_no, "ncluster"].values[0]
      cluster_capacity=df_test.loc[df_test["Test Name"] == test_no, "cluster_capacity"].values[0]

      start_time= perf_counter()
      clf =  KMeans(n_clusters=nCluster, init='k-means++',n_init=10,random_state=seed)

      #clf = KMeansConstrained(n_clusters=nCluster, init='k-means++', size_max=cluster_capacity,random_state=SEED)
      df['assigned_cluster'] = clf.fit_predict(df2)
      end_time=perf_counter()
      df['class']=true_labels

      invalid_constraint=calculate_invalid_constraint(df)
      invalid_capacity=calculate_capacity_violation(df,cluster_capacity)
      # CSV dosyasına yazdırma
      df.to_csv('KmeansConstrained.csv', index=False)
      #print(f"invalid_constraintScore: {invalid_constraint}")
      # Davies-Bouldin Skoru hesapla
      db_score = davies_bouldin_score(df.drop(['class','constraint', 'assigned_cluster'],axis=1), df['assigned_cluster'])
      #print(f"Davies Bouldin Score: {db_score}")
      ch_score = calinski_harabasz_score(df.drop(['class','constraint', 'assigned_cluster'],axis=1), df['assigned_cluster'])
      #print(f"calinski_harabasz_score: {ch_score}")
      dunn_score = dunn_index(df.drop(['class','constraint', 'assigned_cluster'],axis=1), df['assigned_cluster'])
      #print(f"dunn_score: {dunn_score}")

      silhouette1 = silhouette_score(df.drop(['class','constraint', 'assigned_cluster'],axis=1), df['assigned_cluster'])
      #print(f"Silhoutte Score: {silhouette1}")
      ami_score = adjusted_mutual_info_score(df['class'], df['assigned_cluster'])
      #print(f"AMI Score: {ami_score}")
      ari_score = adjusted_rand_score(df['class'], df['assigned_cluster'])
      sse_score = calculate_sse(df.drop(['class','constraint','assigned_cluster'], axis=1),df['assigned_cluster'])
      #print(f"ARI Score: {ari_score}")
      #print(test_no)
      kmeans_constraint_result.append([test_no,run_no,nCluster,percentage,(end_time-start_time),db_score,ch_score,dunn_score,silhouette1,ami_score,ari_score,invalid_constraint,invalid_capacity,sse_score,seed])
kmeans_constraint_result_df=pd.DataFrame(kmeans_constraint_result,columns=result_column_list)

for col in result_column_list:
   if col not in (['TestNo','Run_No','nCluster','Percentage_Constraint']):
      kmeans_constraint_result_df[col] = pd.to_numeric(kmeans_constraint_result_df[col], errors='coerce').round(4)

pivot_df = kmeans_constraint_result_df.pivot_table(
    index=['TestNo', 'Percentage_Constraint'],
    values=['RunTime','Davies-Bouldin','Calinski-Harabasz','Dunn','Silhoutte','Adjusted Mutual Information (AMI)','Adjusted Rand Index (ARI)','invalid_constraint_rate','invalid_capacity_rate', 'sse'],
    aggfunc=['mean', 'std', 'min', 'max']
)
pivot_numeric_columns = pivot_df.select_dtypes(include=['number']).columns

pivot_df[pivot_numeric_columns] = pivot_df[pivot_numeric_columns].round(4)
# Sonuçları bir Excel dosyasına kaydediyoruz
pivot_df.to_excel("revizyon1_pivot_kmeans_results_with_stats.xlsx")
# Pivot DataFrame'deki sayısal sütunları virgülden sonra 4 basamağa yuvarlama

kmeans_constraint_result_df.to_excel("revizyon1_kmeans_results.xlsx")

In [ ]:
#KMEANS CONSTRANT

df_test=pd.read_excel('/content/drive/MyDrive/Real_Test/Real_ConsCluster_Test_Character.xlsx',skiprows=1)
df_test = df_test.iloc[:, 1:]
kmeans_constraint_result=[]
result_column_list=['TestNo','Run_No','nCluster','Percentage_Constraint','RunTime','Davies-Bouldin','Calinski-Harabasz','Dunn','Silhoutte','Adjusted Mutual Information (AMI)','Adjusted Rand Index (ARI)','invalid_constraint_rate','invalid_capacity_rate','sse','seed']
#kmeans_constraint_result.append(result_column_list)
SEEDS = list(range(30))

for test_no in dataset_list:
  for percentage in percentage_list:
    for run_no in range(1,n_run+1):
      seed = SEEDS[run_no - 1]

      df = pd.read_csv('/content/drive/MyDrive/Test/Real_Test_v2/modified_data_ver2/'+str(test_no)+'_data_'+str(percentage)+'.csv')
      true_labels=df['class']
      df.drop(columns=['class'],inplace=True)
      df2 = df.drop(['constraint'], axis=1)
      nCluster= df_test.loc[df_test["Test Name"] == test_no, "ncluster"].values[0]
      cluster_capacity=df_test.loc[df_test["Test Name"] == test_no, "cluster_capacity"].values[0]

      start_time= perf_counter()
      clf = KMeansConstrained(n_clusters=nCluster, init='k-means++', size_max=cluster_capacity,random_state=seed)

      #clf = KMeansConstrained(n_clusters=nCluster, init='k-means++', size_max=cluster_capacity,random_state=SEED)
      df['assigned_cluster'] = clf.fit_predict(df2)
      end_time=perf_counter()
      df['class']=true_labels

      invalid_constraint=calculate_invalid_constraint(df)
      invalid_capacity=calculate_capacity_violation(df,cluster_capacity)
      # CSV dosyasına yazdırma
      #df.to_csv('KmeansConstrained.csv', index=False)
      #print(f"invalid_constraintScore: {invalid_constraint}")
      # Davies-Bouldin Skoru hesapla
      db_score = davies_bouldin_score(df.drop(['class','constraint', 'assigned_cluster'],axis=1), df['assigned_cluster'])
      #print(f"Davies Bouldin Score: {db_score}")
      ch_score = calinski_harabasz_score(df.drop(['class','constraint', 'assigned_cluster'],axis=1), df['assigned_cluster'])
      #print(f"calinski_harabasz_score: {ch_score}")
      dunn_score = dunn_index(df.drop(['class','constraint', 'assigned_cluster'],axis=1), df['assigned_cluster'])
      #print(f"dunn_score: {dunn_score}")

      silhouette1 = silhouette_score(df.drop(['class','constraint', 'assigned_cluster'],axis=1), df['assigned_cluster'])
      #print(f"Silhoutte Score: {silhouette1}")
      ami_score = adjusted_mutual_info_score(df['class'], df['assigned_cluster'])
      #print(f"AMI Score: {ami_score}")
      ari_score = adjusted_rand_score(df['class'], df['assigned_cluster'])
      sse_score = calculate_sse(df.drop(['class', 'constraint', 'assigned_cluster'], axis=1), df['assigned_cluster'])
      #print(f"ARI Score: {ari_score}")
      #print(test_no)
      kmeans_constraint_result.append([test_no,run_no,nCluster,percentage,(end_time-start_time),db_score,ch_score,dunn_score,silhouette1,ami_score,ari_score,invalid_constraint,invalid_capacity,sse_score,seed])
kmeans_constraint_result_df=pd.DataFrame(kmeans_constraint_result,columns=result_column_list)

for col in result_column_list:
   if col not in (['TestNo','Run_No','nCluster','Percentage_Constraint']):
      kmeans_constraint_result_df[col] = pd.to_numeric(kmeans_constraint_result_df[col], errors='coerce').round(4)

pivot_df = kmeans_constraint_result_df.pivot_table(
    index=['TestNo', 'Percentage_Constraint'],
    values=['RunTime','Davies-Bouldin','Calinski-Harabasz','Dunn','Silhoutte','Adjusted Mutual Information (AMI)','Adjusted Rand Index (ARI)','invalid_constraint_rate','invalid_capacity_rate','sse'],
    aggfunc=['mean', 'std', 'min', 'max']
)
pivot_numeric_columns = pivot_df.select_dtypes(include=['number']).columns

pivot_df[pivot_numeric_columns] = pivot_df[pivot_numeric_columns].round(4)
# Sonuçları bir Excel dosyasına kaydediyoruz
pivot_df.to_excel("revizyon1_pivot_kmeans_constraint_results_with_stats.xlsx")
# Pivot DataFrame'deki sayısal sütunları virgülden sonra 4 basamağa yuvarlama

kmeans_constraint_result_df.to_excel("revizyon1_kmeans_constraint_results.xlsx")

In [ ]:
#KMEANSCONSTRAINED+MIN DISTANCE SWAP TEST

df_test=pd.read_excel('/content/drive/MyDrive/Real_Test/Real_ConsCluster_Test_Character.xlsx',skiprows=1)
df_test = df_test.iloc[:, 1:]
kmeans_consmindistance_result=[]
result_column_list=['TestNo','Run_No','nCluster','Percentage_Constraint','RunTime','Davies-Bouldin','Calinski-Harabasz','Dunn','Silhoutte','Adjusted Mutual Information (AMI)','Adjusted Rand Index (ARI)','invalid_constraint_rate','invalid_capacity_rate','sse','seed']

#kmeans_consmindistance_result.append(result_column_list)

SEEDS = list(range(30))
for test_no in dataset_list:
  for percentage in percentage_list:
    for run_no in range(1,n_run+1):
      seed = SEEDS[run_no - 1]
      df = pd.read_csv('/content/drive/MyDrive/Test/Real_Test_v2/modified_data_ver2/'+str(test_no)+'_data_'+str(percentage)+'.csv')
      true_labels=df['class']
      df.drop(columns=['class'],inplace=True)
      df2 = df.drop(columns=['constraint'])


    # KMeansConstrained ile kümeleme

      nCluster= df_test.loc[df_test["Test Name"] == test_no, "ncluster"].values[0]
      cluster_capacity=df_test.loc[df_test["Test Name"] == test_no, "cluster_capacity"].values[0]


      start_time=time.time()
      clf = KMeansConstrained(n_clusters=nCluster, init='k-means++', size_max=cluster_capacity,random_state=seed)

      #clf = KMeansConstrained(n_clusters=nCluster, init='k-means++', size_max=cluster_capacity, random_state=SEED)
      df['assigned_cluster']= clf.fit_predict(df2)
      df['class']=true_labels

      # Küme etiketlerini veri çerçevesine ekleme
      df['id'] = list(range(len(df)))

      # Küme başına araç sahibi sayısını hesaplama
      df3 = df.groupby(['assigned_cluster'])['constraint'].sum().reset_index()

      # Araç sahibi olmayan kümeleri kontrol et
      for cluster_id in range(nCluster):
          cluster_points = df[df['assigned_cluster'] == cluster_id]
          car_owner_count = cluster_points['constraint'].sum()

          if car_owner_count == 0:
              print(f"Küme {cluster_id} araç sahibi içermiyor. Swap başlatılıyor...")

              # Küme merkezini hesapla
              cluster_center = cluster_points.drop(columns=['constraint','class', 'assigned_cluster','id']).mean().values.reshape(1, -1)

              # En az 2 araç sahibi olan kümeleri filtrele
              valid_clusters = df[df.groupby('assigned_cluster')['constraint'].transform('sum') >= 2]

              # Her kümedeki araç sahibi olan elemanları ve mesafeleri hesapla
              min_distance = float('inf')
              swap_candidate = None

              for valid_cluster_id in valid_clusters['assigned_cluster'].unique():
                  candidates = df[(df['assigned_cluster'] == valid_cluster_id) & (df['constraint'] == 1)]
                  if len(candidates) == 0:
                      continue
                  candidate_points = candidates.drop(columns=['constraint', 'class','assigned_cluster','id']).values
                  distances = cdist(cluster_center, candidate_points, metric='euclidean')

                  # En yakın elemanı bul
                  closest_idx = np.argmin(distances)
                  if distances[0, closest_idx] < min_distance:
                      min_distance = distances[0, closest_idx]
                      swap_candidate = candidates.iloc[closest_idx]

              # Swap işlemi
              if swap_candidate is not None:
                  no_owner_point = cluster_points.iloc[0]  # İlk veri noktası
                  df.loc[df['id'] == no_owner_point['id'], 'assigned_cluster'] = swap_candidate['assigned_cluster']
                  df.loc[df['id'] == swap_candidate['id'], 'assigned_cluster'] = cluster_id

                  print(f"Swap işlemi tamamlandı: {no_owner_point['id']} ve {swap_candidate['id']}")

      end_time=time.time()
      # Davies Bouldin ve Silhouette skorları
      df2_for_score = df.drop(columns=['class','constraint', 'assigned_cluster', 'id'])
      db_score = davies_bouldin_score(df2_for_score, df['assigned_cluster'])
      print("Davies Bouldin Score:", db_score)

      silhouette1 = silhouette_score(df2_for_score, df['assigned_cluster'])
      print("Silhoutte Score:", silhouette1)
      ch_score = calinski_harabasz_score(df2_for_score, df['assigned_cluster'])
      dunn_score = dunn_index(df2_for_score, df['assigned_cluster'])
      invalid_constraint=calculate_invalid_constraint(df)
      invalid_capacity=calculate_capacity_violation(df,cluster_capacity)


      ami_score = adjusted_mutual_info_score(df['class'], df['assigned_cluster'])
      print(f"AMI Score: {ami_score}")
      ari_score = adjusted_rand_score(df['class'], df['assigned_cluster'])
      sse_score = calculate_sse( df.drop(columns=['class', 'constraint', 'assigned_cluster', 'id']  ),df['assigned_cluster'])
      print(f"ARI Score: {ari_score}")
      # Sonuçları CSV'ye kaydetme
      #df.to_csv('mindistance_correction2.csv', index=False)
      print(f"calinski_harabasz_score: {ch_score}")
      print(f"dunn_score: {dunn_score}")
      print(f"Davies Bouldin Skoru: {db_score}")
      print(f"Silhoutte Score: {silhouette1}")
      print(f"invalid_constraintScore: {invalid_constraint}")
      kmeans_consmindistance_result.append([test_no,run_no,nCluster,percentage,(end_time-start_time),db_score,ch_score,dunn_score,silhouette1,ami_score,ari_score,invalid_constraint,invalid_capacity,sse_score,seed])


kmeans_consmindistance_result_df=pd.DataFrame(kmeans_consmindistance_result,columns=result_column_list)

for col in result_column_list:
   print(col)
   if col not in (['TestNo','Run_No','nCluster','Percentage_Constraint']):
      kmeans_consmindistance_result_df[col] = pd.to_numeric(kmeans_consmindistance_result_df[col], errors='coerce').round(4)

pivot_df = kmeans_consmindistance_result_df.pivot_table(
    index=['TestNo', 'Percentage_Constraint'],
    values=['RunTime','Davies-Bouldin','Calinski-Harabasz','Dunn','Silhoutte','Adjusted Mutual Information (AMI)','Adjusted Rand Index (ARI)','invalid_constraint_rate','invalid_capacity_rate','sse'],
    aggfunc=['mean', 'std', 'min', 'max']
)
pivot_numeric_columns = pivot_df.select_dtypes(include=['number']).columns

pivot_df[pivot_numeric_columns] = pivot_df[pivot_numeric_columns].round(4)
# Sonuçları bir Excel dosyasına kaydediyoruz
pivot_df.to_excel("revizyon1_pivot_kmeans_consmindistanceswap_results_with_stats.xlsx")
kmeans_consmindistance_result_df.to_excel("revizyon1__kmeans_consmindistanceswap_results.xlsx")



In [ ]:
#KMEANSCONSTRAINED+RANDOM SWAP TEST

df_test=pd.read_excel('/content/drive/MyDrive/Real_Test/Real_ConsCluster_Test_Character.xlsx',skiprows=1)
df_test = df_test.iloc[:, 1:]
result_column_list=['TestNo','Run_No','nCluster','Percentage_Constraint','RunTime','Davies-Bouldin','Calinski-Harabasz','Dunn','Silhoutte','Adjusted Mutual Information (AMI)','Adjusted Rand Index (ARI)','invalid_constraint_rate','invalid_capacity_rate','sse','seed']
kmeans_consrandomswap_result=[]
SEEDS = list(range(30))
#kmeans_consrandomswap_result.append(result_column_list)
for test_no in dataset_list:
  for percentage in percentage_list:
    for run_no in range(1,n_run+1):
      seed = SEEDS[run_no - 1]
      random.seed(seed)
      df = pd.read_csv('/content/drive/MyDrive/Test/Real_Test_v2/modified_data_ver2/'+str(test_no)+'_data_'+str(percentage)+'.csv')
      true_labels=df['class']
      df.drop(columns=['class'],inplace=True)
      df2 = df.drop(columns=['constraint'])

    # KMeansConstrained ile kümeleme


      nCluster= df_test.loc[df_test["Test Name"] == test_no, "ncluster"].values[0]
      cluster_capacity=df_test.loc[df_test["Test Name"] == test_no, "cluster_capacity"].values[0]

      car_ownership_column = 'constraint'

    # Car_Ownership sütunu
      column_index = list(df.columns).index(car_ownership_column)

      features = df.columns[df.columns != car_ownership_column]
      df2 = df[features]
      start_time=time.time()
      # KMeansConstrained
      clf = KMeansConstrained(n_clusters=nCluster, init='k-means++', random_state=seed,size_max=cluster_capacity)
      #clf = KMeansConstrained(n_clusters=nCluster, init='k-means++', size_max=cluster_capacity, random_state=SEED)
      df['assigned_cluster'] = clf.fit_predict(df2)
      df['class']=true_labels
      # Küme etiketlerini veri çerçevesine ekleme
      df['id'] = list(range(len(df)))
      column_to_move = 'assigned_cluster'
      df = df[[col for col in df.columns if col != column_to_move] + [column_to_move]]

      print(df.columns)
      # Kümeleme sonuçlarını sıralama
      df = df.sort_values(by="assigned_cluster", ascending=True)
      print("df", df)
      # Küme başına araç sahibi sayısını hesaplama
      df3 = df.groupby(['assigned_cluster'])[car_ownership_column].sum().reset_index()

      print(df.columns)
      # Verisetini listeye çevirme
      list1 = df.values.tolist()

      # Küme listelerini oluşturma
      cluster_list = [[] for _ in range(nCluster)]
      cluster_biglist = [[] for _ in range(nCluster)]
      for i in range(len(cluster_list)):
          for j in range(len(list1)):
              if list1[j][-1] == i:
                  cluster_list[i].append(list1[j][-2])
                  cluster_biglist[i].append(list1[j])

      print(len(cluster_biglist))
      print(len(cluster_biglist[0]))
      # Küme bazında araç sahibi sayısını gösterme
      car_owners = df3[car_ownership_column].to_list()

      # Araç sahipliğini random olarak değiştirme
      sayac = 0
      for i in range(len(car_owners)):
          if car_owners[i] == 0:  # Kümede araç yoksa
              sayac += 1
              migrate_to = i
              rand2 = random.randint(0, len(car_owners) - 1)

              # Araç sahibi olan birini bulana kadar devam et
              while car_owners[rand2] < 2:
                  rand2 = random.randint(0, len(car_owners) - 1)

              migrate_from = rand2

              rand3 = random.randint(0, len(cluster_list[i]) - 1)
              swap1 = cluster_list[migrate_to][rand3]

              # Araç sahibi olan kullanıcıyı seç
              #for k in range(len(cluster_list[migrate_from])):
              #    if cluster_biglist[migrate_from][k][column_index] == 1:
              #        swap2 = cluster_list[migrate_from][k]
              #        break

              must_have_indices = [k for k in range(len(cluster_list[migrate_from])) if cluster_biglist[migrate_from][k][column_index] == 1]

              k = random.choice(must_have_indices)
              swap2 = cluster_list[migrate_from][k]
              cluster_list[migrate_to].append(swap2)
              cluster_list[migrate_from].append(swap1)
              cluster_list[migrate_to].remove(swap1)
              cluster_list[migrate_from].remove(swap2)

              # Değişiklikleri listeye yansıt
              list_a = cluster_biglist[migrate_from][k]
              list_a[-1] = migrate_to
              list_b = cluster_biglist[migrate_to][rand3]
              list_b[-1] = migrate_from
              cluster_biglist[migrate_to].append(list_a)
              cluster_biglist[migrate_from].append(list_b)

              cluster_biglist[migrate_to].remove(list_b)
              cluster_biglist[migrate_from].remove(list_a)

              car_owners[migrate_to] += 1
              car_owners[migrate_from] -= 1
              print(sayac)

      end_time=time.time()
      # Sonuçları DataFrame olarak kaydet
      df_new = pd.DataFrame(columns=list(features) + [car_ownership_column,'class', 'id', 'assigned_cluster'])

      for m in range(len(cluster_biglist)):
          for n in range(len(cluster_biglist[m])):
              df_new.loc[len(df_new)] = cluster_biglist[m][n]
      print(df_new)
      df_new = df_new.drop(['id'], axis=1)
      df_new.to_csv('random_correction.csv')

      # Küme etiketlerinin tipini integer yap
      print("denew_",df_new)
      df_new['assigned_cluster'] = df_new['assigned_cluster'].astype(int)
      print(df_new['assigned_cluster'])
      df_new_score = df_new.drop(columns=['class','constraint', 'assigned_cluster'])
      # Davies-Bouldin Skoru hesapla
      db_score = davies_bouldin_score(df_new_score, df_new['assigned_cluster'])
      print("Davies Bouldin Score:", db_score)

      # Silhouette Skoru hesapla
      silhouette1 = silhouette_score(df_new_score, df_new['assigned_cluster'])
      print("Silhouette Score:", silhouette1)

      ch_score = calinski_harabasz_score(df_new_score, df_new['assigned_cluster'])
      dunn_score = dunn_index(df_new_score, df_new['assigned_cluster'])
      invalid_constraint=calculate_invalid_constraint(df_new)
      invalid_capacity=calculate_capacity_violation(df_new,cluster_capacity)
      ami_score = adjusted_mutual_info_score(df_new['class'], df_new['assigned_cluster'])
      print(f"AMI Score: {ami_score}")
      ari_score = adjusted_rand_score(df_new['class'], df_new['assigned_cluster'])
      print(f"ARI Score: {ari_score}")
      sse_score = calculate_sse( df_new_score, df_new['assigned_cluster'])
      # Sonuçları CSV'ye kaydetme
      print(f"calinski_harabasz_score: {ch_score}")
      print(f"dunn_score: {dunn_score}")
      print(f"Davies Bouldin Skoru: {db_score}")
      print(f"Silhoutte Score: {silhouette1}")
      print(f"invalid_constraintScore: {invalid_constraint}")




      kmeans_consrandomswap_result.append([test_no,run_no,nCluster,percentage,(end_time-start_time),db_score,ch_score,dunn_score,silhouette1,ami_score,ari_score,invalid_constraint,invalid_capacity,sse_score,seed])
kmeans_consrandomswap_result_df=pd.DataFrame(kmeans_consrandomswap_result,columns=result_column_list)
for col in result_column_list:
   if col not in (['TestNo','Run_No','nCluster','Percentage_Constraint']):
      kmeans_consrandomswap_result_df[col] = pd.to_numeric(kmeans_consrandomswap_result_df[col], errors='coerce').round(4)

pivot_df = kmeans_consrandomswap_result_df.pivot_table(
    index=['TestNo', 'Percentage_Constraint'],
    values=['RunTime','Davies-Bouldin','Calinski-Harabasz','Dunn','Silhoutte','Adjusted Mutual Information (AMI)','Adjusted Rand Index (ARI)','invalid_constraint_rate','invalid_capacity_rate','sse'],
    aggfunc=['mean', 'std', 'min', 'max']
)
pivot_numeric_columns = pivot_df.select_dtypes(include=['number']).columns

pivot_df[pivot_numeric_columns] = pivot_df[pivot_numeric_columns].round(4)
# Sonuçları bir Excel dosyasına kaydediyoruz
pivot_df.to_excel("revizyon1_pivot_kmeans_consrandomswap_results_with_stats.xlsx")
kmeans_consrandomswap_result_df.to_excel("revizyon1_kmeans_consrandomswap_results.xlsx")


In [ ]:
# ============================================================
# GREEDY TEST - FIXED-k + POST-PROCESS REDISTRIBUTION
# Same seeds and SSE calculation as the other algorithms
# ============================================================

df_test = pd.read_excel(
    '/content/drive/MyDrive/Real_Test/Real_ConsCluster_Test_Character.xlsx',
    skiprows=1
)
df_test = df_test.iloc[:, 1:]

kmeans_greedy_result = []

result_column_list = [
    'TestNo',
    'Run_No',
    'nCluster',
    'Percentage_Constraint',
    'RunTime',
    'Davies-Bouldin',
    'Calinski-Harabasz',
    'Dunn',
    'Silhoutte',
    'Adjusted Mutual Information (AMI)',
    'Adjusted Rand Index (ARI)',
    'invalid_constraint_rate',
    'invalid_capacity_rate',
    'sse',
    'seed'
]

SEEDS = list(range(30))


# ============================================================
# GREEDY HELPER FUNCTIONS
# ============================================================

def greedy_compute_centroid(cluster):
    return np.mean(
        [p[:-2] for p in cluster],
        axis=0
    )


def greedy_cluster_sse(cluster):
    """
    SSE of a single cluster.
    Last two columns = class and constraint, therefore excluded.
    """
    if len(cluster) <= 1:
        return 0.0

    X = np.asarray(
        [p[:-2] for p in cluster],
        dtype=float
    )

    center = X.mean(axis=0)

    return float(
        np.sum((X - center) ** 2)
    )


def greedy_delta_sse(point, cluster):
    """
    Increase in SSE if point is added to cluster.
    """
    n = len(cluster)

    x = np.asarray(
        point[:-2],
        dtype=float
    )

    center = greedy_compute_centroid(
        cluster
    )

    squared_dist = np.sum(
        (x - center) ** 2
    )

    return float(
        (n / (n + 1)) * squared_dist
    )


def greedy_assign_min_delta_sse(
    point,
    clusters,
    cluster_capacity
):
    """
    Assign point to the capacity-feasible cluster
    causing the minimum increase in SSE.
    """

    best_idx = None
    best_delta = float('inf')

    for idx, cluster in enumerate(clusters):

        if len(cluster) >= cluster_capacity:
            continue

        delta = greedy_delta_sse(
            point,
            cluster
        )

        if delta < best_delta:
            best_delta = delta
            best_idx = idx

    if best_idx is None:
        return False

    clusters[best_idx].append(point)

    return True


def merge_closest_feasible_clusters_until_k(
    clusters,
    target_k,
    cluster_capacity
):
    """
    Merge closest feasible clusters while possible.
    If no feasible merge exists before target_k is reached,
    stop and continue with post-processing.
    """

    while len(clusters) > target_k:

        feasible_pairs = []

        for i in range(len(clusters)):

            for j in range(
                i + 1,
                len(clusters)
            ):

                if (
                    len(clusters[i])
                    + len(clusters[j])
                    <= cluster_capacity
                ):

                    center_i = greedy_compute_centroid(
                        clusters[i]
                    )

                    center_j = greedy_compute_centroid(
                        clusters[j]
                    )

                    distance = np.linalg.norm(
                        center_i - center_j
                    )

                    feasible_pairs.append(
                        (distance, i, j)
                    )

        if len(feasible_pairs) == 0:
            break

        feasible_pairs.sort(
            key=lambda x: x[0]
        )

        _, i, j = feasible_pairs[0]

        clusters[i].extend(
            clusters[j]
        )

        clusters.pop(j)

    return clusters


def postprocess_redistribute_until_k(
    clusters,
    target_k,
    cluster_capacity
):
    """
    Post-processing step.

    If the merge phase stops with more than target_k clusters:
    - select the cluster with the lowest SSE,
    - eliminate it,
    - redistribute all its points to capacity-feasible clusters
      using minimum Delta SSE,
    - repeat until exactly target_k clusters remain.
    """

    while len(clusters) > target_k:

        # ----------------------------------------------------
        # Select lowest-SSE cluster
        # ----------------------------------------------------

        sse_values = [
            greedy_cluster_sse(cluster)
            for cluster in clusters
        ]

        source_idx = int(
            np.argmin(sse_values)
        )

        source_cluster = list(
            clusters[source_idx]
        )

        # ----------------------------------------------------
        # Verify sufficient free capacity before removal
        # ----------------------------------------------------

        remaining_capacity = sum(
            cluster_capacity - len(cluster)
            for idx, cluster in enumerate(clusters)
            if idx != source_idx
        )

        if remaining_capacity < len(source_cluster):
            raise RuntimeError(
                "Post-processing failed: "
                "insufficient remaining capacity."
            )

        # Remove selected cluster
        clusters.pop(source_idx)

        # ----------------------------------------------------
        # Redistribute points
        # ----------------------------------------------------

        for point in source_cluster:

            assigned = (
                greedy_assign_min_delta_sse(
                    point,
                    clusters,
                    cluster_capacity
                )
            )

            if not assigned:
                raise RuntimeError(
                    "Post-processing failed: "
                    "a point could not be reassigned."
                )

    return clusters


# ============================================================
# MAIN GREEDY EXPERIMENT
# ============================================================

for test_no in dataset_list:

    for percentage in percentage_list:

        for run_no in range(1, n_run + 1):

            seed = SEEDS[run_no - 1]

            # Keep the same seed structure as other algorithms
            random.seed(seed)
            np.random.seed(seed)

            # ------------------------------------------------
            # READ DATA
            # ------------------------------------------------

            df = pd.read_csv(
                '/content/drive/MyDrive/Test/'
                'Real_Test_v2/'
                'modified_data_ver2/'
                + str(test_no)
                + '_data_'
                + str(percentage)
                + '.csv'
            )

            feature_cols = [
                col
                for col in df.columns
                if col not in [
                    'constraint',
                    'class'
                ]
            ]

            car_owner_1 = (
                df[
                    df['constraint'] == 1.0
                ][
                    feature_cols
                    + ['class']
                    + ['constraint']
                ]
                .values
                .tolist()
            )

            car_owner_0 = (
                df[
                    df['constraint'] == 0.0
                ][
                    feature_cols
                    + ['class']
                    + ['constraint']
                ]
                .values
                .tolist()
            )

            # Same k and capacity source as other algorithms
            nCluster = int(
                df_test.loc[
                    df_test["Test Name"] == test_no,
                    "ncluster"
                ].values[0]
            )

            target_k = nCluster

            cluster_capacity = int(
                df_test.loc[
                    df_test["Test Name"] == test_no,
                    "cluster_capacity"
                ].values[0]
            )

            # ------------------------------------------------
            # FEASIBILITY CONDITIONS
            # ------------------------------------------------

            if len(car_owner_1) < target_k:
                raise ValueError(
                    f"{test_no}-{percentage}: "
                    f"must-have count ({len(car_owner_1)}) "
                    f"is smaller than k ({target_k})."
                )

            if len(df) > target_k * cluster_capacity:
                raise ValueError(
                    f"{test_no}-{percentage}: "
                    "total capacity is insufficient."
                )

            # ------------------------------------------------
            # START TIMER
            # ------------------------------------------------

            start_time = time.time()

            # =================================================
            # STEP 1
            # One initial cluster per must-have point
            # =================================================

            clusters = [
                [point]
                for point in car_owner_1
            ]

            # =================================================
            # STEP 2
            # Merge closest feasible clusters
            # =================================================

            clusters = (
                merge_closest_feasible_clusters_until_k(
                    clusters,
                    target_k,
                    cluster_capacity
                )
            )

            # =================================================
            # STEP 3
            # POST-PROCESS if merge cannot reach k
            # =================================================

            if len(clusters) > target_k:

                clusters = (
                    postprocess_redistribute_until_k(
                        clusters,
                        target_k,
                        cluster_capacity
                    )
                )

            if len(clusters) != target_k:

                raise RuntimeError(
                    f"{test_no}-{percentage}: "
                    f"expected {target_k} clusters, "
                    f"obtained {len(clusters)}."
                )

            # =================================================
            # STEP 4
            # Assign non-must-have points by minimum Delta SSE
            # =================================================

            for point in car_owner_0:

                assigned = (
                    greedy_assign_min_delta_sse(
                        point,
                        clusters,
                        cluster_capacity
                    )
                )

                if not assigned:

                    raise RuntimeError(
                        f"{test_no}-{percentage}: "
                        "a non-must-have point "
                        "could not be assigned."
                    )

            end_time = time.time()

            # =================================================
            # CONVERT TO DATAFRAME
            # =================================================

            final_data = []

            for cluster_id, cluster in enumerate(
                clusters
            ):

                for point in cluster:

                    final_data.append(
                        point[:-2]
                        + [
                            point[-2],
                            point[-1],
                            cluster_id
                        ]
                    )

            columns = (
                feature_cols
                + [
                    'class',
                    'constraint',
                    'assigned_cluster'
                ]
            )

            result_df = pd.DataFrame(
                final_data,
                columns=columns
            )

            nCluster_final = (
                result_df[
                    'assigned_cluster'
                ].nunique()
            )

            # =================================================
            # HARD FEASIBILITY CHECKS
            # =================================================

            if nCluster_final != target_k:
                raise RuntimeError(
                    "Exact-k condition violated."
                )

            invalid_constraint = (
                calculate_invalid_constraint(
                    result_df
                )
            )

            invalid_capacity = (
                calculate_capacity_violation(
                    result_df,
                    cluster_capacity
                )
            )

            if invalid_constraint != 0:
                raise RuntimeError(
                    "Must-have constraint violated."
                )

            if invalid_capacity != 0:
                raise RuntimeError(
                    "Capacity constraint violated."
                )

            if len(result_df) != len(df):
                raise RuntimeError(
                    "Not all observations were assigned."
                )

            # =================================================
            # METRICS
            # =================================================

            X_score = result_df[
                feature_cols
            ]

            labels = result_df[
                'assigned_cluster'
            ]

            db_score = (
                davies_bouldin_score(
                    X_score,
                    labels
                )
            )

            ch_score = (
                calinski_harabasz_score(
                    X_score,
                    labels
                )
            )

            dunn_score = (
                dunn_index(
                    X_score,
                    labels
                )
            )

            silhouette1 = (
                silhouette_score(
                    X_score,
                    labels
                )
            )

            ami_score = (
                adjusted_mutual_info_score(
                    result_df['class'],
                    labels
                )
            )

            ari_score = (
                adjusted_rand_score(
                    result_df['class'],
                    labels
                )
            )

            # =================================================
            # SAME SSE FUNCTION AS OTHER METHODS
            # =================================================

            sse_score = calculate_sse(
                X_score,
                labels
            )

            # =================================================
            # SAVE RESULT
            # =================================================

            kmeans_greedy_result.append(
                [
                    test_no,
                    run_no,
                    nCluster_final,
                    percentage,
                    end_time - start_time,
                    db_score,
                    ch_score,
                    dunn_score,
                    silhouette1,
                    ami_score,
                    ari_score,
                    invalid_constraint,
                    invalid_capacity,
                    sse_score,
                    seed
                ]
            )


# ============================================================
# RESULT DATAFRAME
# ============================================================

kmeans_greedy_result_df = pd.DataFrame(
    kmeans_greedy_result,
    columns=result_column_list
)

for col in result_column_list:

    if col not in [
        'TestNo',
        'Run_No',
        'nCluster',
        'Percentage_Constraint'
    ]:

        kmeans_greedy_result_df[col] = (
            pd.to_numeric(
                kmeans_greedy_result_df[col],
                errors='coerce'
            )
            .round(4)
        )


# ============================================================
# PIVOT RESULTS
# ============================================================

pivot_df = (
    kmeans_greedy_result_df
    .pivot_table(
        index=[
            'TestNo',
            'Percentage_Constraint'
        ],
        values=[
            'RunTime',
            'Davies-Bouldin',
            'Calinski-Harabasz',
            'Dunn',
            'Silhoutte',
            'Adjusted Mutual Information (AMI)',
            'Adjusted Rand Index (ARI)',
            'invalid_constraint_rate',
            'invalid_capacity_rate',
            'sse'
        ],
        aggfunc=[
            'mean',
            'std',
            'min',
            'max'
        ]
    )
)

pivot_numeric_columns = (
    pivot_df
    .select_dtypes(
        include=['number']
    )
    .columns
)

pivot_df[
    pivot_numeric_columns
] = (
    pivot_df[
        pivot_numeric_columns
    ]
    .round(4)
)


# ============================================================
# SAVE
# ============================================================

pivot_df.to_excel(
    "revizyon1_pivot_greedy_postprocess_results_with_stats.xlsx"
)

kmeans_greedy_result_df.to_excel(
    "revizyon1_greedy_postprocess_results.xlsx",
    index=False
)


# ============================================================
# FINAL CONTROL
# ============================================================

print(
    "All exact-k:",
    (
        kmeans_greedy_result_df['nCluster']
        == kmeans_greedy_result_df['TestNo'].map(
            df_test.set_index('Test Name')['ncluster']
        )
    ).all()
)

print(
    "All must-have feasible:",
    (
        kmeans_greedy_result_df[
            'invalid_constraint_rate'
        ] == 0
    ).all()
)

print(
    "All capacity feasible:",
    (
        kmeans_greedy_result_df[
            'invalid_capacity_rate'
        ] == 0
    ).all()
)